In [1]:
from pathlib import Path
from neuralnetwork import NeuralNetwork
from semanticreasoning import ProcessData
import torch
import torch.nn as nn
import os
import pandas as pd
import numpy as np
import csv
from torch.utils.data import DataLoader

c:\ProgramData\anaconda3\envs\robotics\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
scores_net = nn.Sequential(
    nn.Linear(23,100),
    nn.ELU(),
    nn.Linear(100,23),
)

value_net = nn.Sequential(
    nn.Linear(23, 100),
    nn.ELU(),
    nn.Linear(100, 23)
)

model = nn.Sequential(
    nn.Linear(407, 500),
    nn.Sigmoid(),
    nn.Linear(500, 500),
    nn.Sigmoid(),
    nn.Linear(500, 500),
    nn.Sigmoid(),
    nn.Linear(500, 100),
    nn.Sigmoid(),
    nn.Linear(100, 19)
)

In [3]:
# Loading Data

processor = ProcessData()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6068.28it/s]


In [4]:
robots = ["AnymalB", "AnymalC", "Badger", "UnitreeA1", "UnitreeGo1", "UnitreeGo2", "UnitreeH1", "RobotisOP3", "Cassie"]

In [5]:
#building the description data 
desc_vector_dict = {}
for robot in robots:

    with open(f"description_vectors/{robot}_description_vectors.csv", "r") as file:
        
        reader = csv.reader(file)

        next(reader)

        list_vectors = []
        for row in reader:
            vector = []
            for vect in row[1:]:
                vector.append((float(vect)))
            list_vectors.append(vector)
                

        desc_vector_dict[robot] = torch.tensor(np.array(list_vectors, dtype=np.float32))


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
with open("training_data.csv", "r") as file:
    file_data = csv.reader(file)

    next(file_data)

    final_dataset = []
    for row in file_data:
        dataset = []
        text = row[0]
        embeddings = processor.getEmbeddings(text)
        robot = row[1]
        other = [float(value) for value in row[2:]]
        dataset = [embeddings,robot]
        dataset.extend(other)
        final_dataset.append(dataset)

In [8]:
dataloader = DataLoader(final_dataset, batch_size=64, shuffle=True)

In [9]:

optimizer = torch.optim.Adam(list(scores_net.parameters())+list(value_net.parameters())+list(model.parameters()), lr=1e-3)
loss_fn = torch.nn.MSELoss()

for epoch in range(2000):
    total_loss = 0
    for batch in dataloader:

        y = torch.stack(batch[2:], dim=1).float() #stack
        robot_names = batch[1]
        #get embeddings
        embeddings = batch[0]
        robot_names = batch[1]
        latent_list = []
        for robot in robot_names:
            #compute latent
            scores = scores_net(desc_vector_dict[robot])
            weights = torch.softmax(scores, dim=-1)

            values = value_net(desc_vector_dict[robot])

            latent = torch.multiply(values, weights)
            latent_sum = latent.sum(dim=0)
            latent_list.append(latent_sum)
        
        embeddings = torch.tensor(embeddings)
        latent_list = torch.stack(latent_list)
        batch_input = torch.cat([embeddings, latent_list], dim=1)
        output = model(batch_input)
        
        loss = loss_fn(output, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss = total_loss+loss.item()
    if (epoch+1)%100 == 0:
        avg_loss = total_loss/len(dataloader)
        print(f"Epoch [{epoch+1}/{5000}], Loss: {avg_loss:.4f}")
    

C:\Users\hp\AppData\Local\Temp\ipykernel_20860\3752509909.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  embeddings = torch.tensor(embeddings)


Epoch [100/5000], Loss: 0.0236
Epoch [200/5000], Loss: 0.0135
Epoch [300/5000], Loss: 0.0060
Epoch [400/5000], Loss: 0.0043
Epoch [500/5000], Loss: 0.0011
Epoch [600/5000], Loss: 0.0006
Epoch [700/5000], Loss: 0.0004
Epoch [800/5000], Loss: 0.0004
Epoch [900/5000], Loss: 0.0002
Epoch [1000/5000], Loss: 0.0002
Epoch [1100/5000], Loss: 0.0002
Epoch [1200/5000], Loss: 0.0001
Epoch [1300/5000], Loss: 0.0001
Epoch [1400/5000], Loss: 0.0001
Epoch [1500/5000], Loss: 0.0001
Epoch [1600/5000], Loss: 0.0001
Epoch [1700/5000], Loss: 0.0001
Epoch [1800/5000], Loss: 0.0001
Epoch [1900/5000], Loss: 0.0000
Epoch [2000/5000], Loss: 0.0002


In [10]:
total_loss

0.0023807342222426087

In [17]:
_text = "stop"
_robot = "AnymalB"
_desc_vector = desc_vector_dict[_robot]

_embedding = processor.getEmbeddings(_text)
_embedding = torch.tensor(_embedding)

scores_net.eval()
value_net.eval()
model.eval()

with torch.no_grad():
    _scores = scores_net(_desc_vector)
    _weights = torch.softmax(_scores, dim=-1)

    _values = value_net(_desc_vector)

    _latent = torch.multiply(_values, _weights)
    _latent_sum = _latent.sum(dim=0)

    _input = torch.cat([_embedding, _latent_sum])


In [18]:
with torch.no_grad():
    _output = model(_input)

In [19]:
_output

tensor([-0.0144,  0.0149,  0.0441, -0.0314,  0.0898,  0.1306, -0.0967,  0.0885,
         0.2038, -0.0971,  0.1417,  0.3250,  0.0086,  0.0359,  0.2746,  0.1171,
         0.0314,  0.1913,  0.9621])

In [21]:
# torch.save(scores_net.state_dict(), "score_nn.pth")
# torch.save(value_net.state_dict(), "value_nn.pth")
# torch.save(model.state_dict(), "main_nn.pth")
